# Laboratorio 2: Armado de un esquema de aprendizaje automático

En el laboratorio final se espera que puedan poner en práctica los conocimientos adquiridos en el curso, trabajando con un conjunto de datos de clasificación.

El objetivo es que se introduzcan en el desarrollo de un esquema para hacer tareas de aprendizaje automático: selección de un modelo, ajuste de hiperparámetros y evaluación.

El conjunto de datos a utilizar está en `./data/loan_data.csv`. Si abren el archivo verán que al principio (las líneas que empiezan con `#`) describen el conjunto de datos y sus atributos (incluyendo el atributo de etiqueta o clase).

Se espera que hagan uso de las herramientas vistas en el curso. Se espera que hagan uso especialmente de las herramientas brindadas por `scikit-learn`.

In [52]:
import numpy as np
import pandas as pd

# TODO: Agregar las librerías que hagan falta
from sklearn.model_selection import train_test_split

## Carga de datos y división en entrenamiento y evaluación

La celda siguiente se encarga de la carga de datos (haciendo uso de pandas). Estos serán los que se trabajarán en el resto del laboratorio.

In [53]:
#dataset = pd.read_csv("./data/loan_data.csv", comment="#")
dataset = pd.read_csv("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#")

# División entre instancias y etiquetas
X, y = dataset.iloc[:, 1:], dataset.TARGET

# división entre entrenamiento y evaluación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


Documentación:

- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

## Ejercicio 1: Descripción de los Datos y la Tarea

Responder las siguientes preguntas:

1. ¿De qué se trata el conjunto de datos?
2. ¿Cuál es la variable objetivo que hay que predecir? ¿Qué significado tiene?
3. ¿Qué información (atributos) hay disponible para hacer la predicción?
4. ¿Qué atributos imagina ud. que son los más determinantes para la predicción?

## Respuesta - Ejercicio 1

\

1. El dataset contiene $5960$ registros de clientes que solicitaron una línea de crédito sobre su valor de propiedad (_home equity_)

\

***
\

2. La variable objetivo que hay que predecir es el incumplimiento de pago de la deuda en cuestión: `TARGET`. Su valor es $1$ si el cliente incumplió su pago y $0$ si este cumplió.

\

***
\

3. Atributos disponibles:

\

| Nombre  | Descripción |
| ------- | ----------- |
| LOAN    | Monto solicitado por el cliente (*) |
| MORTDUE | Monto adeudado en hipoteca actual |
| VALUE   | Valor de la propiedad |
| YOJ     | Cantidad de años en el trabajo actual |
| DEROG   | Cantidad de reportes negativos en el historial crediticio (*) |
| DELINQ  | Cantidad de tardanzas/moras previas en cuentas de crédito (*) |
| CLAGE   | Antigüedad (en meses) de la línea de crédito |
| NINQ    | Número de líneas de crédito abiertas recientemente |
| CLNO    | Número de líneas de crédito |
| DEBTINC | Ratio de deuda e ingreso (*) |

\

***

\

4. Los marcados con `(*)` en la tabla anterior son aquellos los que considero más determinantes para la predicción.

## Ejercicio 2: Predicción con Modelos Lineales

En este ejercicio se entrenarán modelos lineales de clasificación para predecir la variable objetivo.

Para ello, deberán utilizar la clase SGDClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/sgd.html
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html


### Ejercicio 2.1: SGDClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador SGDClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

## Respuesta - Ejercicio 2.1

In [54]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Escalamos y dividimos los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0)

# Modelo con hiperparámetros por defecto
model = SGDClassifier(random_state=0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

def evaluar_modelo(y_true, y_pred, conjunto=""):
    print(f"\nResultados en el conjunto ⇒ {conjunto}\n")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1-score :", f1_score(y_true, y_pred))
    print("Matriz de confusión:")
    print(confusion_matrix(y_true, y_pred))

y_train_pred = model.predict(X_train)
evaluar_modelo(y_train, y_train_pred, conjunto="entrenamiento")

evaluar_modelo(y_test, y_pred, conjunto="evaluación")


Resultados en el conjunto ⇒ entrenamiento

Accuracy : 0.8604180714767363
Precision: 0.6264367816091954
Recall   : 0.4342629482071713
F1-score : 0.5129411764705882
Matriz de confusión:
[[1167   65]
 [ 142  109]]

Resultados en el conjunto ⇒ evaluación

Accuracy : 0.8598382749326146
Precision: 0.575
Recall   : 0.39655172413793105
F1-score : 0.46938775510204084
Matriz de confusión:
[[296  17]
 [ 35  23]]


### Ejercicio 2.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del SGDClassifier. Como mínimo, probar diferentes funciones de loss, tasas de entrenamiento y tasas de regularización.

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

## Respuesta - Ejercicio 2.2

In [55]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Definir el grid de hiperparámetros
param_grid = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],  # función de pérdida
    'alpha': [0.0001, 0.001, 0.01],                   # regularización L2
    'learning_rate': ['constant', 'optimal', 'invscaling'],  # tipo de tasa de aprendizaje
    'eta0': [0.01, 0.1, 1.0]                          # tasa inicial de aprendizaje
}

# Se prueban en total 81 combinaciones de hiperparámetros
grid = GridSearchCV(estimator=model,
                    param_grid=param_grid,
                    scoring='accuracy',
                    cv=5,
                    verbose=1,
                    n_jobs=-1)

grid.fit(X_train, y_train)

results = pd.DataFrame(grid.cv_results_)
results = results[['mean_test_score', 'std_test_score', 'params']]
results_sorted = results.sort_values(by='mean_test_score', ascending=False)

# Mejor estimador encontrado
best_model = grid.best_estimator_
print("\n Mejor combinación de hiperparámetros:")
print(grid.best_params_)

# Evaluación en entrenamiento
y_train_pred = best_model.predict(X_train)
# Evaluación en test
y_test_pred = best_model.predict(X_test)

evaluar_modelo(y_train, y_train_pred, "entrenamiento")
evaluar_modelo(y_test, y_test_pred, "evaluación")


Fitting 5 folds for each of 81 candidates, totalling 405 fits

 Mejor combinación de hiperparámetros:
{'alpha': 0.0001, 'eta0': 1.0, 'learning_rate': 'invscaling', 'loss': 'hinge'}

Resultados en el conjunto ⇒ entrenamiento

Accuracy : 0.8718813216453135
Precision: 0.8279569892473119
Recall   : 0.30677290836653387
F1-score : 0.4476744186046512
Matriz de confusión:
[[1216   16]
 [ 174   77]]

Resultados en el conjunto ⇒ evaluación

Accuracy : 0.8840970350404312
Precision: 0.8947368421052632
Recall   : 0.29310344827586204
F1-score : 0.44155844155844154
Matriz de confusión:
[[311   2]
 [ 41  17]]


## Ejercicio 3: Árboles de Decisión

En este ejercicio se entrenarán árboles de decisión para predecir la variable objetivo.

Para ello, deberán utilizar la clase DecisionTreeClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/tree.html
  - https://scikit-learn.org/stable/modules/tree.html#tips-on-practical-use
- https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html

### Ejercicio 3.1: DecisionTreeClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador DecisionTreeClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


## Respuesta - Ejercicio 3.1

In [56]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0)

tree_model = DecisionTreeClassifier(random_state=0)
tree_model.fit(X_train, y_train)

y_pred_train = tree_model.predict(X_train)
y_pred_test = tree_model.predict(X_test)

evaluar_modelo(y_train, y_pred_train, "entrenamiento")
evaluar_modelo(y_test, y_pred_test, "evaluación")



Resultados en el conjunto ⇒ entrenamiento

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0
Matriz de confusión:
[[1232    0]
 [   0  251]]

Resultados en el conjunto ⇒ evaluación

Accuracy : 0.8867924528301887
Precision: 0.6290322580645161
Recall   : 0.6724137931034483
F1-score : 0.65
Matriz de confusión:
[[290  23]
 [ 19  39]]


### Ejercicio 3.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del DecisionTreeClassifier. Como mínimo, probar diferentes criterios de partición (criterion), profundidad máxima del árbol (max_depth), y cantidad mínima de samples por hoja (min_samples_leaf).

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html